# QuantJourney SDK - Pre-Trade Liquidity and Capacity Check

This notebook demonstrates a QuantJourney SDK workflow that combines prices, dollar volume, short-interest context, sector metadata and valuation fields into a trade-capacity screen.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
universe = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'AVGO', 'JPM', 'LLY', 'XOM']
prices, volumes = price_panel(universe, start='2023-01-01', end=END)
sectors_raw = qj.yf.get_sp500_sectors()
screener_raw = qj.fmp.get_stock_screener(marketCapMoreThan=10000000000, limit=100)
short_raw = {symbol: qj.finra.get_short_interest(symbol=symbol) for symbol in universe[:6]}


In [ ]:
ret = returns(prices)
adv = dollar_adv(prices, volumes).iloc[-1]
vol_63d = ret.tail(63).std() * np.sqrt(252)
ratio_rows = []
for symbol in universe:
    row = unwrap(qj.fmp.get_financial_ratios_ttm(symbol=symbol))
    if isinstance(row, list):
        row = row[0] if row else {}
    ratio_rows.append({'symbol': symbol, 'pe_ttm': pd.to_numeric(row.get('peRatioTTM'), errors='coerce'), 'gross_margin_ttm': pd.to_numeric(row.get('grossProfitMarginTTM'), errors='coerce')})
ratios = pd.DataFrame(ratio_rows).set_index('symbol')


In [ ]:
order_size = 25000000
capacity = pd.DataFrame({'adv_usd': adv, 'capacity_5pct_adv': adv * 0.05, 'capacity_10pct_adv': adv * 0.1, 'volatility_63d': vol_63d, 'momentum_126d': prices.pct_change(126).iloc[-1]}).join(ratios)
capacity['days_to_trade_10pct_adv'] = order_size / capacity['capacity_10pct_adv']
capacity['liquidity_flag'] = np.where(capacity['days_to_trade_10pct_adv'] > 3, 'stagger', 'ok')
capacity['short_feed_rows'] = pd.Series({symbol: len(as_rows(payload)) for symbol, payload in short_raw.items()})
display(capacity.sort_values('days_to_trade_10pct_adv', ascending=False))
capacity['days_to_trade_10pct_adv'].sort_values(ascending=False).plot(kind='bar', title='Days to trade at 10% ADV')
plt.ylabel('days')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.